In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# 1. Fetch Hugging Face token from Kaggle Secrets
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

# 2. Download Phase 7 Prior Case Retrieval (PCR) data
path = snapshot_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=["Phase_7_PCR/**"],
    local_dir="/kaggle/working",
)

# 3. Verify paths and inspect sample structure across Train, Dev, and Test
base_dir = os.path.join("/kaggle/working", "Phase_7_PCR")
files_to_check = {
    "Train Queries": os.path.join(base_dir, "Train", "train.json"),
    "Train Candidates": os.path.join(base_dir, "Train", "train_candidates.json"),
    "Dev Queries": os.path.join(base_dir, "Dev", "dev.json"),
    "Dev Candidates": os.path.join(base_dir, "Dev", "dev_candidates.json"),
    "Test Queries": os.path.join(base_dir, "Test", "test.json"),
    "Test Candidates": os.path.join(base_dir, "Test", "test_candidates.json"),
}

for name, fpath in files_to_check.items():
    if not os.path.exists(fpath):
        print(f"⚠️ {name}: Not found at {fpath}")
        continue
        
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    print(f"\n=== {name} ===")
    print(f"Count: {len(data)}")
    
    if isinstance(data, list) and len(data) > 0:
        sample = data[0]
        if isinstance(sample, dict):
            print("Keys:", list(sample.keys()))
            for k, v in sample.items():
                if isinstance(v, list):
                    preview = f"list(len={len(v)}) -> sample item: {str(v[0])[:80] if v else 'empty'}"
                else:
                    preview = str(v)[:80]
                print(f"  {k}: {preview}")
        else:
            print("Sample element:", str(sample)[:120])
            
    elif isinstance(data, dict) and len(data) > 0:
        first_k = next(iter(data.keys()))
        first_v = data[first_k]
        print(f"Sample Key: {first_k!r}")
        if isinstance(first_v, dict):
            print("Inner Keys:", list(first_v.keys()))
            for k, v in first_v.items():
                if isinstance(v, list):
                    preview = f"list(len={len(v)}) -> sample item: {str(v[0])[:80] if v else 'empty'}"
                else:
                    preview = str(v)[:80]
                print(f"  {k}: {preview}")
        else:
            print(f"Sample Value Type: {type(first_v)}, Preview: {str(first_v)[:100]}")

In [ ]:
import json
import numpy as np

def compute_pcr_metrics_at_7(predictions: dict, ground_truth: dict):
    """
    Computes Micro-F1@7, Micro-Precision@7, and Micro-Recall@7 (Section 13.4).
    predictions: {query_id: [cand_id_1, ..., cand_id_7]}
    ground_truth: {query_id: {"relevant_candidates": [...]}}
    """
    total_intersection = 0
    total_retrieved = 0
    total_gold = 0

    for qid, gold_item in ground_truth.items():
        gold_cands = set(gold_item["relevant_candidates"])
        pred_cands = predictions.get(qid, [])[:7]  # Strictly top 7

        assert len(pred_cands) == 7, f"Query {qid} has {len(pred_cands)} predictions instead of 7!"
        assert len(set(pred_cands)) == 7, f"Query {qid} contains duplicate candidate IDs!"

        total_intersection += len(set(pred_cands).intersection(gold_cands))
        total_retrieved += 7
        total_gold += len(gold_cands)

    micro_precision = total_intersection / total_retrieved if total_retrieved > 0 else 0.0
    micro_recall = total_intersection / total_gold if total_gold > 0 else 0.0
    if (micro_precision + micro_recall) > 0:
        micro_f1 = (2.0 * micro_precision * micro_recall) / (micro_precision + micro_recall)
    else:
        micro_f1 = 0.0

    return {
        "Micro-Precision@7": round(micro_precision * 100.0, 3),
        "Micro-Recall@7": round(micro_recall * 100.0, 3),
        "Micro-F1@7": round(micro_f1 * 100.0, 3),
    }

# Quick sanity check on Dev ground truth
with open("/kaggle/working/Phase_7_PCR/Dev/dev.json", "r", encoding="utf-8") as f:
    dev_gold = json.load(f)

gold_lens = [len(v["relevant_candidates"]) for v in dev_gold.values()]
print(f"Dev Queries: {len(dev_gold)} | Relevant per query: Min={min(gold_lens)}, Median={int(np.median(gold_lens))}, Max={max(gold_lens)}")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# 1. PEFT Modules: LORMA and IA^3
# =====================================================================
class LORMALinear(nn.Module):
    """Compulsory LORMA: Multiplicative Low-Rank Adaptation y = (I + (alpha/r) B A) W_0 x"""
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)
        out_dim = base_layer.out_features
        self.lorma_A = nn.Parameter(torch.empty(rank, out_dim))
        self.lorma_B = nn.Parameter(torch.zeros(out_dim, rank))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = self.base_layer(x)
        mod = (self.dropout(base_out) @ self.lorma_A.T) @ self.lorma_B.T
        return base_out + mod * self.scaling

class IA3Linear(nn.Module):
    """IA^3 (Section 13.2.1 [13]): Multiplicative activation scaling vector y = l * (W_0 x)"""
    def __init__(self, base_layer: nn.Linear):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.ia3_l = nn.Parameter(torch.ones(base_layer.out_features))

    def forward(self, x):
        return self.base_layer(x) * self.ia3_l

# =====================================================================
# 2. Legal Attention Pooling Head
# =====================================================================
class LegalRetrievalPooler(nn.Module):
    def __init__(self, d_model: int, proj_dim: int = 512):
        super().__init__()
        self.attn_mlp = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.Tanh(),
            nn.Linear(d_model // 4, 1, bias=False)
        )
        self.proj = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, proj_dim, bias=False)
        )

    def forward(self, hidden_states: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        # hidden_states: [B, T, D], pad_mask: [B, T] (1 for valid, 0 for pad)
        mask_f = pad_mask.unsqueeze(-1).float()
        mean_pool = (hidden_states * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-6)

        scores = self.attn_mlp(hidden_states).squeeze(-1)
        scores = scores.masked_fill(pad_mask == 0, -1e9)
        alpha = F.softmax(scores, dim=-1).unsqueeze(-1)
        attn_pool = (hidden_states * alpha).sum(dim=1)

        combined = torch.cat([mean_pool, attn_pool], dim=-1)
        return F.normalize(self.proj(combined), p=2, dim=-1)

# =====================================================================
# 3. Bi-Encoder Wrapper for Phase 1 Models
# =====================================================================
class Phase1BiEncoder(nn.Module):
    def __init__(self, base_model: nn.Module, d_model: int, proj_dim: int = 512, arch_type: str = "decoder"):
        super().__init__()
        self.base_model = base_model
        self.arch_type = arch_type
        self.pooler = LegalRetrievalPooler(d_model=d_model, proj_dim=proj_dim)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        if attention_mask is None:
            attention_mask = (input_ids != 0).long()

        if self.arch_type == "arch1":
            B, T = input_ids.size()
            pos = torch.arange(0, T, device=input_ids.device).clamp(max=self.base_model.max_len - 1)
            x = self.base_model.token_emb(input_ids) + self.base_model.pos_emb(pos)
            for blk in self.base_model.blocks:
                x = blk(x)
            hidden = self.base_model.final_norm(x)

        elif self.arch_type == "arch2":
            # Use Encoder of Arch2
            x = self.base_model.token_embedding(input_ids)
            hidden = self.base_model.encoder(x)

        elif self.arch_type == "arch3":
            x = self.base_model.token_emb(input_ids)
            for blk in self.base_model.blocks:
                x = blk(x)
            hidden = self.base_model.final_norm(x)

        elif self.arch_type == "arch4":
            x = self.base_model.token_embedding(input_ids)
            for layer in self.base_model.layers:
                x = layer(x)
            hidden = self.base_model.final_norm(x)

        return self.pooler(hidden, attention_mask)

In [ ]:
import os
import re
import gc
import json
import time
import math
import shutil
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer

# =====================================================================
# 1. Fixed Evaluation Function (Supports Both Short & Full Key Names)
# =====================================================================
def evaluate_pcr_at_7(predictions: dict, ground_truth: dict):
    inter, ret, gold = 0, 0, 0
    for qid, g_item in ground_truth.items():
        g_set = set(g_item["relevant_candidates"])
        p_list = predictions.get(qid, [])[:7]
        assert len(p_list) == 7 and len(set(p_list)) == 7, f"Invalid top-7 list for {qid}"
        inter += len(set(p_list).intersection(g_set))
        ret += 7
        gold += len(g_set)
    p = (inter / ret * 100.0) if ret > 0 else 0.0
    r = (inter / gold * 100.0) if gold > 0 else 0.0
    f1 = (2.0 * p * r) / (p + r) if (p + r) > 0 else 0.0
    return {
        "Micro-P@7": round(p, 2),
        "Micro-R@7": round(r, 2),
        "Micro-F1@7": round(f1, 2),
        "Micro-Precision@7": round(p, 2),
        "Micro-Recall@7": round(r, 2),
    }

def package_pcr_zip(predictions_dict: dict, zip_path: str, expected_queries: dict, expected_cands: dict):
    cand_pool = set(expected_cands.keys())
    assert len(predictions_dict) == len(expected_queries), f"Expected {len(expected_queries)} queries, got {len(predictions_dict)}"
    for qid in expected_queries:
        c_list = predictions_dict[qid]
        assert len(c_list) == 7 and len(set(c_list)) == 7, f"Query {qid} must have 7 unique IDs"
        assert all(c in cand_pool for c in c_list), f"Query {qid} has candidate outside pool"

    tmp_json = "/kaggle/working/prediction.json"
    with open(tmp_json, "w", encoding="utf-8") as f:
        json.dump(predictions_dict, f, indent=2)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(tmp_json, arcname="prediction.json")

    with zipfile.ZipFile(zip_path, "r") as zf:
        assert zf.namelist() == ["prediction.json"]
    print(f"✅ Packaged {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB) -> ['prediction.json']")

# =====================================================================
# 2. Ensure Data & Precompute Sparse TF-IDF Matrices Once
# =====================================================================
BASE_DIR = "/kaggle/working/Phase_7_PCR"
TOKENIZER_PATH = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT_PATHS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
}

tok = Tokenizer.from_file(TOKENIZER_PATH)

def fast_condense_case(sentences, max_sents=32):
    if not isinstance(sentences, list):
        sentences = [str(sentences)]
    if len(sentences) <= max_sents:
        return " ".join(sentences)
    h = max_sents // 2
    return " ".join(sentences[:h] + sentences[-(max_sents - h):])

if "train_q_text" not in globals():
    with open(os.path.join(BASE_DIR, "Train", "train.json"), "r") as f: train_queries = json.load(f)
    with open(os.path.join(BASE_DIR, "Train", "train_candidates.json"), "r") as f: train_cands = json.load(f)
    with open(os.path.join(BASE_DIR, "Dev", "dev.json"), "r") as f: dev_queries = json.load(f)
    with open(os.path.join(BASE_DIR, "Dev", "dev_candidates.json"), "r") as f: dev_cands = json.load(f)
    with open(os.path.join(BASE_DIR, "Test", "test.json"), "r") as f: test_queries = json.load(f)
    with open(os.path.join(BASE_DIR, "Test", "test_candidates.json"), "r") as f: test_cands = json.load(f)

    train_q_text = {k: fast_condense_case(v["text"]) for k, v in train_queries.items()}
    train_c_text = {k: fast_condense_case(v["text"]) for k, v in train_cands.items()}
    dev_q_text   = {k: fast_condense_case(v["text"]) for k, v in dev_queries.items()}
    dev_c_text   = {k: fast_condense_case(v["text"]) for k, v in dev_cands.items()}
    test_q_text  = {k: fast_condense_case(v["text"]) for k, v in test_queries.items()}
    test_c_text  = {k: fast_condense_case(v["text"]) for k, v in test_cands.items()}

dev_q_ids = list(dev_q_text.keys())
dev_c_ids = list(dev_c_text.keys())
test_q_ids = list(test_q_text.keys())
test_c_ids = list(test_c_text.keys())
train_c_ids = list(train_c_text.keys())

# Precompute Sparse Lexical Similarity for Dev & Test + Hard Negatives for Train
print("Computing multi-granular word+char TF-IDF matrices for Dev, Test, and Hard Negatives...")
vec_word = TfidfVectorizer(max_features=35000, sublinear_tf=True, stop_words="english", ngram_range=(1, 2))
all_texts = list(train_c_text.values()) + list(dev_c_text.values()) + list(test_c_text.values())
vec_word.fit(all_texts)

dev_sim_sparse = (vec_word.transform([dev_q_text[k] for k in dev_q_ids]) @ vec_word.transform([dev_c_text[k] for k in dev_c_ids]).T).toarray()
test_sim_sparse = (vec_word.transform([test_q_text[k] for k in test_q_ids]) @ vec_word.transform([test_c_text[k] for k in test_c_ids]).T).toarray()

dev_sim_sparse_z = (dev_sim_sparse - dev_sim_sparse.mean(axis=1, keepdims=True)) / (dev_sim_sparse.std(axis=1, keepdims=True) + 1e-6)
test_sim_sparse_z = (test_sim_sparse - test_sim_sparse.mean(axis=1, keepdims=True)) / (test_sim_sparse.std(axis=1, keepdims=True) + 1e-6)

# Mine Top Hard Negatives per Training Query using Sparse Similarity
train_q_list = [q for q in train_queries if len(train_queries[q]["relevant_candidates"]) > 0]
train_sim_mat = (vec_word.transform([train_q_text[k] for k in train_q_list]) @ vec_word.transform([train_c_text[k] for k in train_c_ids]).T).toarray()

hard_neg_map = {}
for idx, qid in enumerate(train_q_list):
    pos_set = set(train_queries[qid]["relevant_candidates"])
    top_indices = np.argsort(train_sim_mat[idx])[::-1][:25]
    negs = [train_c_ids[i] for i in top_indices if train_c_ids[i] not in pos_set]
    hard_neg_map[qid] = negs[:5] if negs else [train_c_ids[0]]
del train_sim_mat
gc.collect()

# =====================================================================
# 3. Pre-Tokenize All Cases ONCE into Tensors (Eliminates 40-Min Bottleneck!)
# =====================================================================
MAX_LEN = 256
print(f"Pre-tokenizing all Train/Dev/Test texts once (max_len={MAX_LEN})...")
t_tok = time.time()

def batch_tokenize_dict(text_dict, max_len=MAX_LEN):
    keys = list(text_dict.keys())
    encoded = tok.encode_batch([text_dict[k] for k in keys], add_special_tokens=False)
    out = {}
    for k, enc in zip(keys, encoded):
        ids = enc.ids
        if len(ids) > max_len:
            ids = ids[:max_len // 2] + ids[-(max_len - max_len // 2):]
        out[k] = torch.tensor(ids if ids else [1], dtype=torch.long)
    return out

train_q_tok = batch_tokenize_dict(train_q_text)
train_c_tok = batch_tokenize_dict(train_c_text)
dev_q_tok   = batch_tokenize_dict(dev_q_text)
dev_c_tok   = batch_tokenize_dict(dev_c_text)
test_q_tok  = batch_tokenize_dict(test_q_text)
test_c_tok  = batch_tokenize_dict(test_c_text)
print(f"Pre-tokenization completed in {time.time() - t_tok:.2f}s!")

# =====================================================================
# 4. All 4 Phase 1 Architectures + Approved Phase 7 PEFT Modules
# =====================================================================
def apply_rope(x, cos, sin):
    d = x.shape[-1] // 2
    return torch.cat([x[..., :d] * cos - x[..., d:] * sin, x[..., d:] * cos + x[..., :d] * sin], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        return freqs.cos().to(dtype)[None, None, :, :], freqs.sin().to(dtype)[None, None, :, :]

# --- Arch 1 (8L Decoder, d=768) ---
class Arch1Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12, max_len=1024):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.register_buffer("mask", torch.tril(torch.ones(1, 1, max_len, max_len)))
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch1MLP(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    def forward(self, x): return self.fc2(F.gelu(self.fc1(x)))

class Arch1Block(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch1Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class Arch1Backbone(nn.Module):
    def __init__(self, vocab_size=40000, d_model=768, n_layers=8):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model)
        self.blocks = nn.ModuleList([Arch1Block(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, input_ids):
        B, T = input_ids.size()
        pos = torch.arange(0, T, device=input_ids.device).clamp(max=1023)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks: x = blk(x)
        return self.final_norm(x)

# --- Arch 2 (6L Bidirectional Encoder from Seq2Seq, d=768) ---
class Arch2Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim)
    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch2FFN(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
    def forward(self, x): return self.linear2(F.gelu(self.linear1(x)))

class Arch2EncLayer(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = Arch2Attention(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch2FFN(d_model)
    def forward(self, x):
        x = x + self.self_attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class Arch2EncoderBackbone(nn.Module):
    def __init__(self, vocab_size=40000, d_model=768, n_layers=6):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.Module()
        self.encoder.layers = nn.ModuleList([Arch2EncLayer(d_model) for _ in range(n_layers)])
    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for layer in self.encoder.layers: x = layer(x)
        return x

# --- Arch 3 (16L SWA, d=1024) ---
class Arch3Block(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, d_ff=4096):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch2Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class Arch3Backbone(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=16):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([Arch3Block(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, input_ids):
        x = self.token_emb(input_ids)
        for blk in self.blocks: x = blk(x)
        return self.final_norm(x)

# --- Approved Phase 7 PEFT Modules: LORMA [12] & IA^3 [13] ---
class LORMALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters(): p.requires_grad = False
        self.scaling = alpha / rank
        self.lorma_A = nn.Parameter(torch.empty(rank, base_layer.out_features))
        self.lorma_B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
    def forward(self, x):
        b = self.base_layer(x)
        return b + ((b @ self.lorma_A.T) @ self.lorma_B.T) * self.scaling

class IA3Linear(nn.Module):
    """IA^3 (Section 13.2.1 [13]): Element-wise multiplicative rescaling y = l * (W_0 x)"""
    def __init__(self, base_layer: nn.Linear):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters(): p.requires_grad = False
        self.ia3_l = nn.Parameter(torch.ones(base_layer.out_features))
    def forward(self, x):
        return self.base_layer(x) * self.ia3_l

class UniversalBiEncoder(nn.Module):
    def __init__(self, backbone: nn.Module, d_model: int, proj_dim: int = 512, pool_type: str = "mean_attn"):
        super().__init__()
        self.backbone = backbone
        self.pool_type = pool_type
        if pool_type == "mean_attn":
            self.attn_w = nn.Linear(d_model, 1, bias=False)
            self.proj = nn.Sequential(nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, proj_dim, bias=False))
        elif pool_type == "max_mean":
            self.proj = nn.Sequential(nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, proj_dim, bias=False))
        else:
            self.proj = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, proj_dim, bias=False))

    def forward(self, input_ids):
        mask = (input_ids != 0).unsqueeze(-1).float()
        h = self.backbone(input_ids)
        mean_p = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        if self.pool_type == "mean_attn":
            scores = self.attn_w(h).squeeze(-1).masked_fill(input_ids == 0, -1e4)
            attn_p = (h * F.softmax(scores, dim=-1).unsqueeze(-1)).sum(dim=1)
            pooled = torch.cat([mean_p, attn_p], dim=-1)
        elif self.pool_type == "max_mean":
            max_p = h.masked_fill(mask == 0, -1e4).max(dim=1).values
            pooled = torch.cat([mean_p, max_p], dim=-1)
        else:
            pooled = mean_p
        return F.normalize(self.proj(pooled), p=2, dim=-1)

# =====================================================================
# 5. Fast Pre-Tokenized Query-Centric Dataset (Trains in ~90s!)
# =====================================================================
class FastQueryContrastiveDataset(Dataset):
    def __init__(self, q_list):
        self.q_list = q_list
    def __len__(self):
        return len(self.q_list)
    def __getitem__(self, idx):
        qid = self.q_list[idx]
        pos_cands = [c for c in train_queries[qid]["relevant_candidates"] if c in train_c_tok]
        pos_cid = pos_cands[np.random.randint(len(pos_cands))]
        neg_cands = hard_neg_map[qid]
        neg_cid = neg_cands[np.random.randint(len(neg_cands))]
        return {
            "q": train_q_tok[qid],
            "pos": train_c_tok[pos_cid],
            "neg": train_c_tok[neg_cid],
        }

def collate_triplets(batch):
    q = nn.utils.rnn.pad_sequence([b["q"] for b in batch], batch_first=True, padding_value=0)
    pos = nn.utils.rnn.pad_sequence([b["pos"] for b in batch], batch_first=True, padding_value=0)
    neg = nn.utils.rnn.pad_sequence([b["neg"] for b in batch], batch_first=True, padding_value=0)
    return {"q": q, "pos": pos, "neg": neg}

@torch.no_grad()
def fast_encode_tok_dict(model, tok_dict, keys_order, batch_size=64):
    model.eval()
    all_embs = []
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    for i in range(0, len(keys_order), batch_size):
        b_keys = keys_order[i:i + batch_size]
        padded = nn.utils.rnn.pad_sequence([tok_dict[k] for k in b_keys], batch_first=True, padding_value=0).cuda()
        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
            emb = model(padded)
        all_embs.append(emb.float().cpu())
    return torch.cat(all_embs, dim=0)

# =====================================================================
# 6. Step A: Rescue Already-Trained Exp 1 (Arch4 + LORMA) or Train Fast
# =====================================================================
results_summary = []
test_sim_matrices = {}
dev_sim_matrices = {}

if "encoder" in globals() and "dev_sim_neural" in globals():
    print("\n🎉 Found already-trained Exp 1 (Arch 4 + LORMA) in memory! Saving immediately...")
    exp1_dev_neural = dev_sim_neural
    dev_q_embs_t = dev_q_embs
    dev_c_embs_t = dev_c_embs
    # Encode Test using pre-tokenized dict (takes 4 seconds!)
    t_q_emb = fast_encode_tok_dict(encoder, test_q_tok, test_q_ids)
    t_c_emb = fast_encode_tok_dict(encoder, test_c_tok, test_c_ids)
    exp1_test_neural = (t_q_emb.cuda() @ t_c_emb.cuda().T).cpu().numpy()
    teacher_model = encoder
else:
    print("\n🚀 Training Exp 1: Arch 4 (Hybrid GQA) + Compulsory LORMA...")
    ckpt = torch.load(CKPT_PATHS["arch4"], map_location="cpu", weights_only=False)
    sd = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items() if "lm_head" not in k}
    bb4 = Arch4HybridGQAModel()
    bb4.load_state_dict(sd, strict=False)
    for p in bb4.parameters(): p.requires_grad = False
    for name, mod in list(bb4.named_modules()):
        if isinstance(mod, nn.Linear):
            p_name, c_name = name.rsplit(".", 1)
            setattr(bb4.get_submodule(p_name), c_name, LORMALinear(mod, rank=16))
    for m in bb4.modules():
        if isinstance(m, nn.LayerNorm):
            for p in m.parameters(): p.requires_grad = True
    teacher_model = UniversalBiEncoder(bb4, d_model=1024, proj_dim=512, pool_type="mean").cuda()
    del ckpt, sd; gc.collect()

    loader = DataLoader(FastQueryContrastiveDataset(train_q_list), batch_size=16, shuffle=True, collate_fn=collate_triplets, drop_last=True)
    opt = torch.optim.AdamW([p for p in teacher_model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')
    teacher_model.train()
    for ep in range(3):
        for batch in loader:
            with torch.amp.autocast('cuda', dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
                q_e = teacher_model(batch["q"].cuda())
                p_e = teacher_model(batch["pos"].cuda())
                n_e = teacher_model(batch["neg"].cuda())
                c_all = torch.cat([p_e, n_e], dim=0)
                loss = F.cross_entropy((q_e @ c_all.T) / 0.05, torch.arange(q_e.size(0), device="cuda"))
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); opt.zero_grad()

    d_q_emb = fast_encode_tok_dict(teacher_model, dev_q_tok, dev_q_ids)
    d_c_emb = fast_encode_tok_dict(teacher_model, dev_c_tok, dev_c_ids)
    exp1_dev_neural = (d_q_emb.cuda() @ d_c_emb.cuda().T).cpu().numpy()
    t_q_emb = fast_encode_tok_dict(teacher_model, test_q_tok, test_q_ids)
    t_c_emb = fast_encode_tok_dict(teacher_model, test_c_tok, test_c_ids)
    exp1_test_neural = (t_q_emb.cuda() @ t_c_emb.cuda().T).cpu().numpy()

# Evaluate & Package Exp 1 (Arch4 + LORMA)
exp1_dev_z = (exp1_dev_neural - exp1_dev_neural.mean(axis=1, keepdims=True)) / (exp1_dev_neural.std(axis=1, keepdims=True) + 1e-6)
exp1_test_z = (exp1_test_neural - exp1_test_neural.mean(axis=1, keepdims=True)) / (exp1_test_neural.std(axis=1, keepdims=True) + 1e-6)

dev_sim_matrices["exp1_arch4_lorma"] = 0.55 * exp1_dev_z + 0.45 * dev_sim_sparse_z
test_sim_matrices["exp1_arch4_lorma"] = 0.55 * exp1_test_z + 0.45 * test_sim_sparse_z

exp1_dev_preds = {qid: [dev_c_ids[i] for i in np.argsort(dev_sim_matrices["exp1_arch4_lorma"][idx])[::-1][:7]] for idx, qid in enumerate(dev_q_ids)}
exp1_test_preds = {qid: [test_c_ids[i] for i in np.argsort(test_sim_matrices["exp1_arch4_lorma"][idx])[::-1][:7]] for idx, qid in enumerate(test_q_ids)}
m1 = evaluate_pcr_at_7(exp1_dev_preds, dev_queries)
print(f"Exp 1 (Arch4 + LORMA) Dev -> Micro-P@7: {m1['Micro-P@7']}% | Micro-R@7: {m1['Micro-R@7']}% | Micro-F1@7: {m1['Micro-F1@7']}%")
package_pcr_zip(exp1_test_preds, "/kaggle/working/submission_exp1_arch4_lorma.zip", test_queries, test_cands)
results_summary.append(("Exp 1", "Arch 4 (242M GQA)", "LORMA (Compulsory)", "Mean Pool + InfoNCE", m1["Micro-P@7"], m1["Micro-R@7"], m1["Micro-F1@7"]))

# =====================================================================
# 7. Runner for Exp 2 (Arch3 + IA^3), Exp 3 (Arch2 + Partial FT), Exp 4 (Arch1 + KD)
# =====================================================================
def run_fast_pcr_experiment(exp_key, exp_label, arch_name, method_name, pool_type, loss_type, teacher=None, epochs=3, lr=4e-4):
    print(f"\n🚀 Running {exp_label}: {arch_name.upper()} + {method_name} ({loss_type})...")
    t0 = time.time()
    ckpt = torch.load(CKPT_PATHS[arch_name], map_location="cpu", weights_only=False)
    sd = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}

    if arch_name == "arch3":
        bb = Arch3Backbone()
        bb.load_state_dict(sd, strict=False)
        d_model = 1024
    elif arch_name == "arch2":
        bb = Arch2EncoderBackbone()
        bb.load_state_dict(sd, strict=False)
        d_model = 768
    elif arch_name == "arch1":
        bb = Arch1Backbone()
        bb.load_state_dict(sd, strict=False)
        d_model = 768
    del ckpt, sd; gc.collect()

    # Apply Approved Section 13.2.1 Technique
    for p in bb.parameters():
        p.requires_grad = False

    if method_name == "IA3":
        for name, mod in list(bb.named_modules()):
            if isinstance(mod, nn.Linear) and any(t in name for t in ("k_proj", "v_proj", "fc2", "out_proj")):
                p_name, c_name = name.rsplit(".", 1)
                setattr(bb.get_submodule(p_name), c_name, IA3Linear(mod))
        for m in bb.modules():
            if isinstance(m, nn.LayerNorm):
                for p in m.parameters(): p.requires_grad = True

    elif method_name == "Partial_Layerwise_FT":
        # Unfreeze top 2 Encoder layers (layers.4 and layers.5) of Arch 2
        for layer in bb.encoder.layers[-2:]:
            for p in layer.parameters():
                p.requires_grad = True

    elif method_name == "Knowledge_Distillation":
        # Student Arch 1 adapted with LORMA + Top Block Unfrozen for fast KD alignment
        for name, mod in list(bb.named_modules()):
            if isinstance(mod, nn.Linear):
                p_name, c_name = name.rsplit(".", 1)
                setattr(bb.get_submodule(p_name), c_name, LORMALinear(mod, rank=16))
        for p in bb.blocks[-1].parameters():
            p.requires_grad = True

    model = UniversalBiEncoder(bb, d_model=d_model, proj_dim=512, pool_type=pool_type).cuda()
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable_p:,} / {total_p:,} ({100*trainable_p/total_p:.2f}%)")

    loader = DataLoader(FastQueryContrastiveDataset(train_q_list), batch_size=16, shuffle=True, collate_fn=collate_triplets, drop_last=True)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    model.train()
    if teacher is not None:
        teacher.eval()

    for ep in range(epochs):
        for batch in loader:
            q_ids = batch["q"].cuda()
            p_ids = batch["pos"].cuda()
            n_ids = batch["neg"].cuda()

            with torch.amp.autocast('cuda', dtype=amp_dtype):
                q_e = model(q_ids)
                p_e = model(p_ids)
                n_e = model(n_ids)
                c_all = torch.cat([p_e, n_e], dim=0)
                sim_s = (q_e @ c_all.T) / 0.05
                labels = torch.arange(q_e.size(0), device="cuda")

                if loss_type == "margin_ranking":
                    pos_sim = (q_e * p_e).sum(dim=-1)
                    neg_sim = (q_e * n_e).sum(dim=-1)
                    margin_loss = F.relu(0.25 - pos_sim + neg_sim).mean()
                    loss = 0.5 * F.cross_entropy(sim_s, labels) + 0.5 * margin_loss

                elif loss_type == "kd" and teacher is not None:
                    with torch.no_grad():
                        tq_e = teacher(q_ids)
                        tp_e = teacher(p_ids)
                        tn_e = teacher(n_ids)
                        tc_all = torch.cat([tp_e, tn_e], dim=0)
                        sim_t = (tq_e @ tc_all.T) / 0.05
                    # Relational Similarity Matrix KL Distillation + InfoNCE
                    kd_loss = F.kl_div(
                        F.log_softmax(sim_s.float(), dim=-1),
                        F.softmax(sim_t.float(), dim=-1),
                        reduction="batchmean"
                    )
                    loss = 0.4 * F.cross_entropy(sim_s, labels) + 0.6 * kd_loss
                else:
                    loss = F.cross_entropy(sim_s, labels)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()

    # Encode Dev & Test in < 10 seconds
    d_q = fast_encode_tok_dict(model, dev_q_tok, dev_q_ids)
    d_c = fast_encode_tok_dict(model, dev_c_tok, dev_c_ids)
    dev_neu = (d_q.cuda() @ d_c.cuda().T).cpu().numpy()

    t_q = fast_encode_tok_dict(model, test_q_tok, test_q_ids)
    t_c = fast_encode_tok_dict(model, test_c_tok, test_c_ids)
    test_neu = (t_q.cuda() @ t_c.cuda().T).cpu().numpy()

    dev_z = (dev_neu - dev_neu.mean(axis=1, keepdims=True)) / (dev_neu.std(axis=1, keepdims=True) + 1e-6)
    test_z = (test_neu - test_neu.mean(axis=1, keepdims=True)) / (test_neu.std(axis=1, keepdims=True) + 1e-6)

    dev_sim_matrices[exp_key] = 0.55 * dev_z + 0.45 * dev_sim_sparse_z
    test_sim_matrices[exp_key] = 0.55 * test_z + 0.45 * test_sim_sparse_z

    dev_preds = {qid: [dev_c_ids[i] for i in np.argsort(dev_sim_matrices[exp_key][idx])[::-1][:7]] for idx, qid in enumerate(dev_q_ids)}
    test_preds = {qid: [test_c_ids[i] for i in np.argsort(test_sim_matrices[exp_key][idx])[::-1][:7]] for idx, qid in enumerate(test_q_ids)}
    m = evaluate_pcr_at_7(dev_preds, dev_queries)
    print(f"  Done in {time.time()-t0:.1f}s! Dev -> Micro-P@7: {m['Micro-P@7']}% | Micro-R@7: {m['Micro-R@7']}% | Micro-F1@7: {m['Micro-F1@7']}%")

    package_pcr_zip(test_preds, f"/kaggle/working/submission_{exp_key}.zip", test_queries, test_cands)
    results_summary.append((exp_label, f"{arch_name.upper()}", method_name, f"{pool_type} + {loss_type}", m["Micro-P@7"], m["Micro-R@7"], m["Micro-F1@7"]))

    del model, bb; gc.collect(); torch.cuda.empty_cache()

# Run Exp 4 (Arch 1 Student KD) FIRST while Teacher Arch 4 is in GPU memory
run_fast_pcr_experiment(
    exp_key="exp4_arch1_kd",
    exp_label="Exp 4 (KD)",
    arch_name="arch1",
    method_name="Knowledge_Distillation",
    pool_type="mean_attn",
    loss_type="kd",
    teacher=teacher_model,
    epochs=4,
    lr=4e-4
)

# Free Teacher Arch 4 before loading Arch 3 and Arch 2
del teacher_model
if "encoder" in globals():
    del encoder
gc.collect()
torch.cuda.empty_cache()

# Run Exp 2 (Arch 3 SWA + IA^3 + Max-Mean Pool + Margin Ranking)
run_fast_pcr_experiment(
    exp_key="exp2_arch3_ia3",
    exp_label="Exp 2",
    arch_name="arch3",
    method_name="IA3",
    pool_type="max_mean",
    loss_type="margin_ranking",
    epochs=3,
    lr=1e-3
)

# Run Exp 3 (Arch 2 Bidirectional Encoder + Partial Layer-wise Fine-Tuning)
run_fast_pcr_experiment(
    exp_key="exp3_arch2_partial_ft",
    exp_label="Exp 3",
    arch_name="arch2",
    method_name="Partial_Layerwise_FT",
    pool_type="mean_attn",
    loss_type="listwise_infonce",
    epochs=4,
    lr=3e-4
)

# =====================================================================
# 8. Build 4-Model Ensemble (submission.zip) & Print Final Report Table
# =====================================================================
ens_dev_sim = sum(dev_sim_matrices.values()) / len(dev_sim_matrices)
ens_test_sim = sum(test_sim_matrices.values()) / len(test_sim_matrices)

ens_dev_preds = {qid: [dev_c_ids[i] for i in np.argsort(ens_dev_sim[idx])[::-1][:7]] for idx, qid in enumerate(dev_q_ids)}
ens_test_preds = {qid: [test_c_ids[i] for i in np.argsort(ens_test_sim[idx])[::-1][:7]] for idx, qid in enumerate(test_q_ids)}

m_ens = evaluate_pcr_at_7(ens_dev_preds, dev_queries)
results_summary.append(("Ensemble", "All 4 Phase 1 Archs", "Multi-Arch Fusion", "Rank Score Aggregation", m_ens["Micro-P@7"], m_ens["Micro-R@7"], m_ens["Micro-F1@7"]))
package_pcr_zip(ens_test_preds, "/kaggle/working/submission.zip", test_queries, test_cands)

print("\n" + "=" * 95)
print(f"{'Exp':<12} | {'Phase 1 Model':<19} | {'Technique (Sec 13.2.1)':<23} | {'P@7 (%)':<8} | {'R@7 (%)':<8} | {'F1@7 (%)':<8}")
print("-" * 95)
for row in results_summary:
    print(f"{row[0]:<12} | {row[1]:<19} | {row[2]:<23} | {row[4]:<8.2f} | {row[5]:<8.2f} | {row[6]:<8.2f}")
print("=" * 95)

In [ ]:
import json
import zipfile
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

print("Building Full-Document Multi-Field Legal Retriever + Graph Diffusion...")

# 1. Use 100% of sentences from every case (No truncation!)
full_train_c = [" ".join(train_cands[cid]["text"]) for cid in train_c_ids]
full_dev_q   = [" ".join(dev_queries[qid]["text"]) for qid in dev_q_ids]
full_dev_c   = [" ".join(dev_cands[cid]["text"]) for cid in dev_c_ids]
full_test_q  = [" ".join(test_queries[qid]["text"]) for qid in test_q_ids]
full_test_c  = [" ".join(test_cands[cid]["text"]) for cid in test_c_ids]

all_full_corpus = full_train_c + full_dev_c + full_test_c

# 2. Granularity 1: Fine-Grained Word Unigrams + Bigrams + Trigrams (Captures exact legal doctrines & Acts)
vec_w13 = TfidfVectorizer(
    max_features=100000,
    sublinear_tf=True,
    stop_words="english",
    ngram_range=(1, 3),
    min_df=2
)
vec_w13.fit(all_full_corpus)

dev_q_w13  = normalize(vec_w13.transform(full_dev_q))
dev_c_w13  = normalize(vec_w13.transform(full_dev_c))
test_q_w13 = normalize(vec_w13.transform(full_test_q))
test_c_w13 = normalize(vec_w13.transform(full_test_c))

sim_dev_w13  = (dev_q_w13 @ dev_c_w13.T).toarray()
sim_test_w13 = (test_q_w13 @ test_c_w13.T).toarray()

# 3. Granularity 2: Statutory & Section-Specific Legal Pattern Extractor
import re
STATUTE_PATTERN = re.compile(
    r"\b(?:section|sec|article|art|order|rule|clause|schedule|act|ipc|crpc|cpc)\s+[0-9a-zA-Z\(\)\-]+",
    re.IGNORECASE
)
def extract_statutes(text):
    matches = STATUTE_PATTERN.findall(text.lower())
    return " ".join(m.replace(" ", "_") for m in matches) if matches else "none"

vec_stat = TfidfVectorizer(max_features=25000, sublinear_tf=True, ngram_range=(1, 2))
vec_stat.fit([extract_statutes(t) for t in all_full_corpus])

sim_dev_stat  = (vec_stat.transform([extract_statutes(t) for t in full_dev_q]) @ vec_stat.transform([extract_statutes(t) for t in full_dev_c]).T).toarray()
sim_test_stat = (vec_stat.transform([extract_statutes(t) for t in full_test_q]) @ vec_stat.transform([extract_statutes(t) for t in full_test_c]).T).toarray()

# 4. Candidate-to-Candidate Graph Diffusion / Pseudo-Relevance Feedback (PRF)
# If Query Q matches Top-5 Candidates strongly, propagate similarity to their nearest legal neighbors
dev_cc_sim  = (dev_c_w13 @ dev_c_w13.T).toarray()
test_cc_sim = (test_c_w13 @ test_c_w13.T).toarray()
np.fill_diagonal(dev_cc_sim, 0.0)
np.fill_diagonal(test_cc_sim, 0.0)

def compute_prf_diffusion(q_c_sim, c_c_sim, top_k=5):
    prf_sim = np.zeros_like(q_c_sim)
    for i in range(q_c_sim.shape[0]):
        top_idx = np.argsort(q_c_sim[i])[::-1][:top_k]
        weights = np.exp(q_c_sim[i, top_idx] * 5.0)
        weights /= weights.sum()
        prf_sim[i] = weights @ c_c_sim[top_idx]
    return prf_sim

sim_dev_prf  = compute_prf_diffusion(sim_dev_w13, dev_cc_sim, top_k=5)
sim_test_prf = compute_prf_diffusion(sim_test_w13, test_cc_sim, top_k=5)

# Z-score normalize each signal per query
def z_norm(mat):
    return (mat - mat.mean(axis=1, keepdims=True)) / (mat.std(axis=1, keepdims=True) + 1e-6)

dev_w13_z, test_w13_z   = z_norm(sim_dev_w13), z_norm(sim_test_w13)
dev_stat_z, test_stat_z = z_norm(sim_dev_stat), z_norm(sim_test_stat)
dev_prf_z, test_prf_z   = z_norm(sim_dev_prf), z_norm(sim_test_prf)

# Extract pure neural component from our best neural models (Exp 3 Arch2 + Exp 4 Arch1 KD)
dev_best_neural_z  = z_norm(0.55 * dev_sim_matrices["exp3_arch2_partial_ft"] + 0.45 * dev_sim_matrices["exp4_arch1_kd"] - 0.45 * dev_sim_sparse_z)
test_best_neural_z = z_norm(0.55 * test_sim_matrices["exp3_arch2_partial_ft"] + 0.45 * test_sim_matrices["exp4_arch1_kd"] - 0.45 * test_sim_sparse_z)

# 5. Grid-Search Optimal Weights on Dev to Maximize Micro-F1@7
best_f1 = -1.0
best_weights = None

for w_lex in [0.55, 0.65, 0.75]:
    for w_stat in [0.10, 0.15, 0.25]:
        for w_prf in [0.05, 0.10, 0.15]:
            for w_neu in [0.10, 0.15, 0.20, 0.25]:
                comb_dev = (w_lex * dev_w13_z) + (w_stat * dev_stat_z) + (w_prf * dev_prf_z) + (w_neu * dev_best_neural_z)
                preds = {qid: [dev_c_ids[i] for i in np.argsort(comb_dev[idx])[::-1][:7]] for idx, qid in enumerate(dev_q_ids)}
                m = evaluate_pcr_at_7(preds, dev_queries)
                if m["Micro-F1@7"] > best_f1:
                    best_f1 = m["Micro-F1@7"]
                    best_weights = (w_lex, w_stat, w_prf, w_neu, m)

w_lex, w_stat, w_prf, w_neu, best_m = best_weights
print(f"\n🔥 Optimal Dev Weights: Lex={w_lex}, Statute={w_stat}, PRF={w_prf}, Neural={w_neu}")
print(f"🔥 Upgraded Dev Scores -> Micro-P@7: {best_m['Micro-P@7']}% | Micro-R@7: {best_m['Micro-R@7']}% | Micro-F1@7: {best_m['Micro-F1@7']}%")

# 6. Generate & Package Boosted Test Submission
comb_test = (w_lex * test_w13_z) + (w_stat * test_stat_z) + (w_prf * test_prf_z) + (w_neu * test_best_neural_z)
boosted_test_preds = {qid: [test_c_ids[i] for i in np.argsort(comb_test[idx])[::-1][:7]] for idx, qid in enumerate(test_q_ids)}

package_pcr_zip(boosted_test_preds, "/kaggle/working/submission_boosted.zip", test_queries, test_cands)

In [ ]:
import re
import json
import zipfile
import scipy.sparse as sp
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

print("Building Ultra-PCR Pipeline: Vectorized BM25 + Case Citation/Year Matcher + Reciprocal Graph Reranking...")

# 1. Full Document Texts
full_train_c = [" ".join(train_cands[cid]["text"]) for cid in train_c_ids]
full_dev_q   = [" ".join(dev_queries[qid]["text"]) for qid in dev_q_ids]
full_dev_c   = [" ".join(dev_cands[cid]["text"]) for cid in dev_c_ids]
full_test_q  = [" ".join(test_queries[qid]["text"]) for qid in test_q_ids]
full_test_c  = [" ".join(test_cands[cid]["text"]) for cid in test_c_ids]
all_full_corpus = full_train_c + full_dev_c + full_test_c

# 2. Vectorized Sparse Okapi BM25 Matrix Scorer (Runs in < 2 seconds!)
def compute_sparse_bm25_similarity(query_texts, cand_texts, fit_corpus, k1=1.6, b=0.75, ngram_range=(1, 2), max_features=120000):
    cv = CountVectorizer(max_features=max_features, stop_words="english", ngram_range=ngram_range, min_df=2)
    cv.fit(fit_corpus)

    X_q = cv.transform(query_texts).tocsr().astype(np.float32)
    X_c = cv.transform(cand_texts).tocsr().astype(np.float32)

    # Compute IDF on candidate pool + fit corpus
    N = X_c.shape[0]
    df = np.bincount(X_c.indices, minlength=X_c.shape[1])
    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)

    # Candidate document lengths
    doc_len = np.array(X_c.sum(axis=1)).flatten()
    avgdl = float(doc_len.mean()) + 1e-6

    # Apply BM25 TF saturation to candidate matrix X_c in-place
    X_c_bm25 = X_c.copy().tocoo()
    norm_factor = k1 * (1.0 - b + b * (doc_len[X_c_bm25.row] / avgdl))
    X_c_bm25.data = (X_c_bm25.data * (k1 + 1.0)) / (X_c_bm25.data + norm_factor)
    X_c_bm25 = X_c_bm25.tocsr()

    # Weight query terms by IDF and sublinear log(1 + tf)
    X_q_log = X_q.copy()
    X_q_log.data = np.log1p(X_q_log.data) * idf[X_q_log.indices]

    return (X_q_log @ X_c_bm25.T).toarray()

sim_dev_bm25  = compute_sparse_bm25_similarity(full_dev_q, full_dev_c, all_full_corpus, k1=1.6, b=0.75, ngram_range=(1, 2))
sim_test_bm25 = compute_sparse_bm25_similarity(full_test_q, full_test_c, all_full_corpus, k1=1.6, b=0.75, ngram_range=(1, 2))

# 3. Sublinear Word + Phrase (1-3 Gram) Cosine Similarity
vec_w13 = TfidfVectorizer(max_features=120000, sublinear_tf=True, stop_words="english", ngram_range=(1, 3), min_df=2)
vec_w13.fit(all_full_corpus)
dev_q_w13, dev_c_w13   = normalize(vec_w13.transform(full_dev_q)), normalize(vec_w13.transform(full_dev_c))
test_q_w13, test_c_w13 = normalize(vec_w13.transform(full_test_q)), normalize(vec_w13.transform(full_test_c))

sim_dev_w13  = (dev_q_w13 @ dev_c_w13.T).toarray()
sim_test_w13 = (test_q_w13 @ test_c_w13.T).toarray()

# 4. Legal Statute, Section, Article & Case-Year Citation Extractor
LEGAL_CIT_REGEX = re.compile(
    r"\b(?:section|sec|s\.|article|art\.|order|rule|clause|schedule|act|ipc|crpc|cpc)\s*[0-9a-zA-Z\(\)\-]+|\b(?:19[4-9]\d|20[0-2]\d)\b",
    re.IGNORECASE
)
def extract_legal_markers(text):
    hits = LEGAL_CIT_REGEX.findall(text.lower())
    return " ".join(re.sub(r"\s+", "_", h.strip()) for h in hits) if hits else "none"

vec_cit = TfidfVectorizer(max_features=35000, sublinear_tf=True, ngram_range=(1, 3))
vec_cit.fit([extract_legal_markers(t) for t in all_full_corpus])
dev_q_cit, dev_c_cit   = normalize(vec_cit.transform([extract_legal_markers(t) for t in full_dev_q])), normalize(vec_cit.transform([extract_legal_markers(t) for t in full_dev_c]))
test_q_cit, test_c_cit = normalize(vec_cit.transform([extract_legal_markers(t) for t in full_test_q])), normalize(vec_cit.transform([extract_legal_markers(t) for t in full_test_c]))

sim_dev_cit  = (dev_q_cit @ dev_c_cit.T).toarray()
sim_test_cit = (test_q_cit @ test_c_cit.T).toarray()

# 5. 2-Hop Graph Diffusion + Reciprocal Nearest-Neighbor (k-NN) Reranking
def z_norm(mat):
    return (mat - mat.mean(axis=1, keepdims=True)) / (mat.std(axis=1, keepdims=True) + 1e-6)

dev_base_lex  = 0.50 * z_norm(sim_dev_bm25) + 0.35 * z_norm(sim_dev_w13) + 0.15 * z_norm(sim_dev_cit)
test_base_lex = 0.50 * z_norm(sim_test_bm25) + 0.35 * z_norm(sim_test_w13) + 0.15 * z_norm(sim_test_cit)

# Candidate-to-Candidate combined legal graph (Lexical + Statutory)
dev_cc_graph  = 0.70 * (dev_c_w13 @ dev_c_w13.T).toarray() + 0.30 * (dev_c_cit @ dev_c_cit.T).toarray()
test_cc_graph = 0.70 * (test_c_w13 @ test_c_w13.T).toarray() + 0.30 * (test_c_cit @ test_c_cit.T).toarray()
np.fill_diagonal(dev_cc_graph, 0.0)
np.fill_diagonal(test_cc_graph, 0.0)

def compute_graph_prf(base_sim, cc_graph, top_k=6, temp=2.5):
    prf = np.zeros_like(base_sim)
    for i in range(base_sim.shape[0]):
        top_idx = np.argsort(base_sim[i])[::-1][:top_k]
        w = np.exp(base_sim[i, top_idx] / temp)
        w /= w.sum()
        prf[i] = w @ cc_graph[top_idx]
    return z_norm(prf)

dev_prf_z  = compute_graph_prf(dev_base_lex, dev_cc_graph, top_k=6)
test_prf_z = compute_graph_prf(test_base_lex, test_cc_graph, top_k=6)

# Neural Ensemble Signal from Phase 1 Fine-Tuned Bi-Encoders (Arch1 KD + Arch2 Partial FT + Arch4 LORMA)
dev_neu_ens = z_norm(
    0.45 * (dev_sim_matrices["exp3_arch2_partial_ft"] - 0.45 * dev_sim_sparse_z) +
    0.40 * (dev_sim_matrices["exp4_arch1_kd"] - 0.45 * dev_sim_sparse_z) +
    0.15 * (dev_sim_matrices["exp1_arch4_lorma"] - 0.45 * dev_sim_sparse_z)
)
test_neu_ens = z_norm(
    0.45 * (test_sim_matrices["exp3_arch2_partial_ft"] - 0.45 * test_sim_sparse_z) +
    0.40 * (test_sim_matrices["exp4_arch1_kd"] - 0.45 * test_sim_sparse_z) +
    0.15 * (test_sim_matrices["exp1_arch4_lorma"] - 0.45 * test_sim_sparse_z)
)

# 6. Coordinate Search on Dev to Maximize Micro-F1@7
best_f1 = -1.0
best_cfg = None

for w_bm25 in [0.35, 0.45, 0.55]:
    for w_w13 in [0.25, 0.35]:
        for w_cit in [0.10, 0.15, 0.20]:
            for w_prf in [0.08, 0.12, 0.18]:
                for w_neu in [0.08, 0.12, 0.16]:
                    comb = (
                        w_bm25 * z_norm(sim_dev_bm25) +
                        w_w13  * z_norm(sim_dev_w13) +
                        w_cit  * z_norm(sim_dev_cit) +
                        w_prf  * dev_prf_z +
                        w_neu  * dev_neu_ens
                    )
                    preds = {qid: [dev_c_ids[i] for i in np.argsort(comb[idx])[::-1][:7]] for idx, qid in enumerate(dev_q_ids)}
                    m = evaluate_pcr_at_7(preds, dev_queries)
                    if m["Micro-F1@7"] > best_f1:
                        best_f1 = m["Micro-F1@7"]
                        best_cfg = (w_bm25, w_w13, w_cit, w_prf, w_neu, m)

w_bm25, w_w13, w_cit, w_prf, w_neu, best_m = best_cfg
print(f"\n🚀 Optimal Ultra Weights: BM25={w_bm25}, W13={w_w13}, Cit={w_cit}, PRF={w_prf}, Neural={w_neu}")
print(f"🚀 Dev Evaluation -> Micro-P@7: {best_m['Micro-P@7']}% | Micro-R@7: {best_m['Micro-R@7']}% | Micro-F1@7: {best_m['Micro-F1@7']}%")

# 7. Package Ultra Test Submission
comb_test = (
    w_bm25 * z_norm(sim_test_bm25) +
    w_w13  * z_norm(sim_test_w13) +
    w_cit  * z_norm(sim_test_cit) +
    w_prf  * test_prf_z +
    w_neu  * test_neu_ens
)
ultra_test_preds = {qid: [test_c_ids[i] for i in np.argsort(comb_test[idx])[::-1][:7]] for idx, qid in enumerate(test_q_ids)}
package_pcr_zip(ultra_test_preds, "/kaggle/working/submission_ultra_pcr.zip", test_queries, test_cands)